# Modelado — ablación sociodemográfica

Comparamos población, tres fracciones y un eje urbano de 2017 con variantes anuales reconstruidas. CP1 se ajusta por fold. Las variantes anuales de 2018–2024 incorporan información del extremo 2025: sus métricas son retrospectivas. La selección usa solo 2021–2023 y separa alternativas fijas de anuales.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modeling.ablacion_sociodemografica import ejecutar_ablacion_sociodemografica

reporte = ejecutar_ablacion_sociodemografica()

h=2: 30160 filas comunes; 325 de arranque excluidas
  base_6: F1 validación reg=0.677, cls=0.598
  fija_poblacion: F1 validación reg=0.674, cls=0.659
  fija_tres: F1 validación reg=0.684, cls=0.591
  fija_eje: F1 validación reg=0.678, cls=0.621
  fija_poblacion_eje: F1 validación reg=0.677, cls=0.633
  anual_poblacion: F1 validación reg=0.675, cls=0.670
  anual_tres: F1 validación reg=0.683, cls=0.590
  anual_eje: F1 validación reg=0.679, cls=0.594
  anual_15: F1 validación reg=0.657, cls=0.603
  anual_poblacion_eje: F1 validación reg=0.665, cls=0.644
  Elegidas h=2: {'fija': {'regresion': 'fija_tres', 'clasificacion': 'fija_poblacion'}, 'anual_retrospectiva': {'regresion': 'anual_tres', 'clasificacion': 'anual_poblacion'}}
h=4: 30030 filas comunes; 455 de arranque excluidas
  base_6: F1 validación reg=0.568, cls=0.557
  fija_poblacion: F1 validación reg=0.561, cls=0.581
  fija_tres: F1 validación reg=0.561, cls=0.547
  fija_eje: F1 validación reg=0.564, cls=0.565
  fija_poblacion_eje:

In [2]:
import pandas as pd

filas = []
for h, bloque in reporte['horizontes'].items():
    elegidas = bloque['elegidas_solo_validacion']
    for variante, resultado in bloque['variantes'].items():
        folds = resultado['folds_validacion']
        filas.append({'h': int(h), 'variante': variante, 'n_variables': len(resultado['columnas']),
                      'F1_regresion': sum(f['regresion']['alerta_derivada']['f1'] for f in folds) / len(folds),
                      'F1_clasificacion': sum(f['clasificacion']['alerta_directa']['f1'] for f in folds) / len(folds),
                      'elegida_fija': variante in elegidas['fija'].values(),
                      'elegida_anual_retrospectiva': variante in elegidas['anual_retrospectiva'].values()})
display(pd.DataFrame(filas).sort_values(['h', 'variante']))

,h,variante,n_variables,F1_regresion,F1_clasificacion,elegida_fija,elegida_anual_retrospectiva
8,2,anual_15,21,0.657416,0.602768,False,False
7,2,anual_eje,7,0.678525,0.594345,False,False
5,2,anual_poblacion,7,0.674508,0.669579,False,True
9,2,anual_poblacion_eje,8,0.664738,0.643642,False,False
6,2,anual_tres,9,0.683112,0.590106,False,True
0,2,base_6,6,0.676984,0.598093,False,False
3,2,fija_eje,7,0.678261,0.620578,False,False
1,2,fija_poblacion,7,0.674318,0.659480,True,False
4,2,fija_poblacion_eje,8,0.677325,0.633380,False,False
2,2,fija_tres,9,0.683841,0.590934,True,False


In [3]:
filas = []
for h, bloque in reporte['horizontes'].items():
    for variante, resultado in bloque['variantes'].items():
        for fold in resultado['folds_prueba']:
            for enfoque, metrica in (('regresion', fold['regresion']['alerta_derivada']),
                                     ('clasificacion', fold['clasificacion']['alerta_directa'])):
                filas.append({'h': int(h), 'bloque': fold['bloque'], 'variante': variante,
                              'enfoque': enfoque, 'positivos': metrica['positivos'],
                              'VP': metrica['vp'], 'FP': metrica['fp'], 'F1': metrica['f1']})
display(pd.DataFrame(filas).sort_values(['h', 'bloque', 'enfoque', 'variante']))

,h,bloque,variante,enfoque,positivos,VP,FP,F1
15,2,calendario_2025,anual_poblacion,clasificacion,3,2,163,0.023810
19,2,calendario_2025,anual_tres,clasificacion,3,2,375,0.010526
3,2,calendario_2025,base_6,clasificacion,3,2,320,0.012308
7,2,calendario_2025,fija_poblacion,clasificacion,3,2,161,0.024096
11,2,calendario_2025,fija_tres,clasificacion,3,1,273,0.007220
14,2,calendario_2025,anual_poblacion,regresion,3,0,3,0.000000
18,2,calendario_2025,anual_tres,regresion,3,0,3,0.000000
2,2,calendario_2025,base_6,regresion,3,0,3,0.000000
6,2,calendario_2025,fija_poblacion,regresion,3,0,3,0.000000
10,2,calendario_2025,fija_tres,regresion,3,0,3,0.000000


El informe `docs/modeling/ablacion_sociodemografica.md` separa los resultados fijos de las reconstrucciones anuales. Los bloques 2024 y 2025 no participaron en la selección de variables.